In [5]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
import plotly.graph_objects as go
import warnings

In [6]:
# Suppress warnings for cleaner output (e.g., division by zero during early optimization steps)
warnings.filterwarnings('ignore')

# Set plotting style
pd.options.display.float_format = '{:,.4f}'.format

## The SABR Model
The implied volatility $\sigma_{imp}(K)$ is approximated by the Hagan et al. (2002) expansion:

$$
\sigma(K) = \frac{\alpha z}{\chi(z)} \cdot \left\{ 1 + \left[ \frac{2-3\rho^2}{24}\nu^2 \right] T \right\}
$$

Where:
* $z = \frac{\nu}{\alpha} (F - K)$
* $\chi(z) = \ln \left( \frac{\sqrt{1 - 2\rho z + z^2} + z - \rho}{1 - \rho} \right)$

In [7]:
def sabr_vol_normal(k, f, t, alpha, rho, nu):
    """
    Hagan et al. (2002) Normal SABR Approximation.
    Returns implied Normal Volatility (absolute, not lognormal).
    """
    # 1. Handle ATM case (f == k) to avoid division by zero
    if abs(f - k) < 1e-8:
        term1 = alpha
        term2 = (2 - 3 * rho**2) / 24 * nu**2 * t
        return term1 * (1 + term2)
    
    # 2. General case
    z = (nu / alpha) * (f - k)
    
    # Safety: Ensure term inside log is positive
    # chi(z) = log( (sqrt(1 - 2*rho*z + z^2) + z - rho) / (1 - rho) )
    discriminant = 1 - 2*rho*z + z**2
    numerator = np.sqrt(discriminant) + z - rho
    denominator = 1 - rho
    
    if numerator <= 0 or denominator == 0:
        return alpha # Fallback to Alpha if parameters go rogue
        
    chi_z = np.log(numerator / denominator)
    
    if abs(chi_z) < 1e-8: # Safety for extremely small chi
        return alpha
        
    multiplier = alpha * (z / chi_z)
    bracket = 1 + ((2 - 3 * rho**2) / 24 * nu**2) * t
    
    return multiplier * bracket

def bachelier_price(f, k, t, vol, is_call=True):
    """
    Bachelier (Normal) Model Pricing Formula.
    """
    d = (f - k) / (vol * np.sqrt(t))
    
    if is_call:
        price = (f - k) * norm.cdf(d) + vol * np.sqrt(t) * norm.pdf(d)
    else:
        price = (k - f) * norm.cdf(-d) + vol * np.sqrt(t) * norm.pdf(d)
        
    return price

In [8]:
# Load the cleaned "Tidy" data
# Expected columns: Expiry, Tenor, Forward, Annuity, ATM_Vol_Price, Strike_Offset, Payer_Price, Receiver_Price
try:
    df = pd.read_csv('../data/processed/swaption_data_clean.csv')
    print("✅ Data loaded successfully.")
    print(df.head())
except FileNotFoundError:
    print("❌ Error: Processed data not found. Please run '01_Data_Cleaning.ipynb' first.")

✅ Data loaded successfully.
  Expiry Tenor  Forward  Annuity  Offset  Straddle  Strangle  RiskReversal  \
0     3m    1y   0.0342   0.9581  0.0025        20        10            -2   
1     6m    1y   0.0333   0.9506  0.0050        31        13            -3   
2     1y    1y   0.0332   0.9350  0.0100        50        18            -3   
3     2y    1y   0.0350   0.9044  0.0100        78        42             1   
4     3y    1y   0.0366   0.9647  0.0150        96        46             3   

    Payer  Receiver  Strike_Payer  Strike_Receiver  T_expiry  
0  4.0000    6.0000        0.0367           0.0317    0.2500  
1  5.0000    8.0000        0.0383           0.0283    0.5000  
2  7.5000   10.5000        0.0432           0.0232    1.0000  
3 21.5000   20.5000        0.0450           0.0250    2.0000  
4 24.5000   21.5000        0.0516           0.0216    3.0000  


### Calibration

In [11]:
# Prepare a list to store results
calibration_results = []

# Group by Expiry and Tenor (Calibrate one swaption at a time)
groups = df.groupby(['Expiry', 'Tenor'])

print(f"Starting calibration for {len(groups)} swaptions...")

for (expiry_label, tenor_label), group in groups:
    
    # Extract market constants for this swaption
    # Assumption: Forward, Annuity, T_expiry are constant across the row
    F = group['Forward'].iloc[0]
    A = group['Annuity'].iloc[0]
    T = group['T_expiry'].iloc[0]
    
    # Target Prices (Market)
    # We calibrate to 3 instruments: ATM Straddle, OTM Payer, OTM Receiver
    # Note: Divide by 10,000 if your CSV prices are still in bps!
    # Here we assume prices are already DECIMALS.
    P_straddle = group['Straddle'].iloc[0]
    P_payer = group['Payer'].iloc[0]
    P_receiver = group['Receiver'].iloc[0]
    
    # Strikes
    K_atm = F
    K_payer = group['Strike_Payer'].iloc[0]
    K_receiver = group['Strike_Receiver'].iloc[0]
    
    # --- Objective Function ---
    def objective(params):
        alpha, rho, nu = params
        
        # 1. Hard Constraints (Penalties)
        if alpha <= 0 or nu <= 0 or abs(rho) >= 0.999:
            return 1e10 # Huge error for invalid params
            
        # 2. Calculate Model Vols
        vol_atm = sabr_vol_normal(K_atm, F, T, alpha, rho, nu)
        vol_pay = sabr_vol_normal(K_payer, F, T, alpha, rho, nu)
        vol_rec = sabr_vol_normal(K_receiver, F, T, alpha, rho, nu)
        
        # 3. Calculate Model Prices (Bachelier)
        # ATM Straddle = 2 * Call(ATM)
        model_straddle = 2 * bachelier_price(F, K_atm, T, vol_atm, is_call=True) * A
        model_payer    = bachelier_price(F, K_payer, T, vol_pay, is_call=True) * A
        model_receiver = bachelier_price(F, K_receiver, T, vol_rec, is_call=False) * A
        
        # 4. Sum of Squared Errors (SSE)
        error = (model_straddle - P_straddle)**2 + \
                (model_payer - P_payer)**2 + \
                (model_receiver - P_receiver)**2
        return error

    # --- Optimization ---
    # Initial Guess: Alpha ~ Implied Normal Vol, Rho=0, Nu=0.5
    # Approx Normal Vol = Straddle / (2 * A * sqrt(T/2pi))
    vol_guess = P_straddle / (2 * A * np.sqrt(T / (2 * np.pi)))
    initial_guess = [vol_guess, 0.0, 0.5]
    
    result = minimize(objective, initial_guess, method='Nelder-Mead', tol=1e-8)
    
    # Store Results
    calibration_results.append({
        'Expiry': expiry_label,
        'Tenor': tenor_label,
        'T_expiry': T,
        'Alpha': result.x[0],
        'Rho': result.x[1],
        'Nu': result.x[2],
        'Error': result.fun
    })

# Convert to DataFrame
calib_df = pd.DataFrame(calibration_results)

# Sort for better viewing (requires mapping labels to numbers if not done)
print("Calibration Complete!")
print(calib_df.head())

# Save results
calib_df.to_csv('../data/processed/sabr_parameters.csv', index=False)

Starting calibration for 35 swaptions...
Calibration Complete!
  Expiry Tenor  T_expiry   Alpha    Rho     Nu        Error
0    10y   10y   10.0000 49.8514 0.0011 0.5500  65,202.6543
1    10y    1y   10.0000 53.9922 0.0009 0.5430     996.0036
2    10y    2y   10.0000 53.4126 0.0009 0.5429   3,687.9638
3    10y   30y   10.0000 45.4605 0.0011 0.5509 276,035.4104
4    10y    5y   10.0000 52.3620 0.0009 0.5431  20,114.4285


### Visualization

In [12]:
def plot_sabr_surface(data_df, param_name, title, color_scale='Viridis'):
    """
    Generates an interactive 3D surface plot for a given SABR parameter.
    """
    # 1. Pivot Data: Rows=Expiry, Cols=Tenor, Values=Parameter
    # Ensure logical sort order for axis labels
    expiry_order = ['3m', '6m', '1y', '2y', '3y', '5y', '10y']
    tenor_order = ['1y', '2y', '5y', '10y', '30y']
    
    pivot_df = data_df.pivot(index='Expiry', columns='Tenor', values=param_name)
    pivot_df = pivot_df.reindex(expiry_order)[tenor_order]
    
    # 2. Create Grid
    x = tenor_order
    y = expiry_order
    z = pivot_df.values
    
    # 3. Plotly Surface
    fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale=color_scale)])
    
    fig.update_layout(
        title=title,
        scene = dict(
            xaxis_title='Tenor (Swap Length)',
            yaxis_title='Expiry (Option Time)',
            zaxis_title=param_name,
            xaxis=dict(tickmode='array', tickvals=list(range(len(x))), ticktext=x),
            yaxis=dict(tickmode='array', tickvals=list(range(len(y))), ticktext=y),
        ),
        width=800,
        height=600,
        margin=dict(l=65, r=50, b=65, t=90)
    )
    
    fig.show()

# --- Generate the 3 Surfaces ---

# 1. Alpha Surface (ATM Volatility)
plot_sabr_surface(calib_df, 'Alpha', 'Alpha Surface (ATM Volatility)', 'Plasma')

# 2. Rho Surface (Skew)
plot_sabr_surface(calib_df, 'Rho', 'Rho Surface (Skew)', 'RdBu')

# 3. Nu Surface (Vol of Vol)
plot_sabr_surface(calib_df, 'Nu', 'Nu Surface (Vol of Vol)', 'Viridis')